First of all, I'm trying to think which version to use first and what changes to make to it!
what I think I'll do for the first submission is: Partial fine tuning (and better cleaning) the v1 phase 1 submission

# Citations: 
1) Training-free layer selection for partial fine-tuning of language
models



In [ ]:
!pip install ijson
import os
import sys
import json
import math
import time
import random
import zipfile
from typing import List, Tuple, Dict, Any, Generator
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
from tqdm.auto import tqdm
import numpy as np, ijson
from multiprocessing import Pool
import regex

In [ ]:

HF_TOKEN = None
if "HF_TOKEN" in os.environ:
    HF_TOKEN = os.environ["HF_TOKEN"]
else:
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        HF_TOKEN = user_secrets.get_secret("HF")
        os.environ["HF_TOKEN"] = HF_TOKEN
        print("Successfully loaded HF_TOKEN from Kaggle Secrets.")
    except Exception as e:
        print(f"Note: Kaggle secrets not detected or HF key missing ({e}). Will attempt public access.")

if HF_TOKEN:
    try:
        from huggingface_hub import login
        login(token=HF_TOKEN)
        print("Huggingface hub authentication successful.")
    except Exception as err:
        print(f"HF Login warning: {err}")


print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
        
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        print(os.path.join(root, file))

In [ ]:
from huggingface_hub import HfApi

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

HF_TOKENIZER_UPLOAD_REPO = os.environ.get("HF_TOKENIZER_UPLOAD_REPO", "legal-bpe-tokenizer-phase1")



TRAIN_FILES = ["Phase_2_LNER/Train/fold1-gold.json", "Phase_2_LNER/Train/fold1-text.json",
              "Phase_2_LNER/Train/fold2-text.json", "Phase_2_LNER/Train/fold2-gold.json"]

TEST_FILES = ["Phase_2_LNER/Test/fold3-text.json"]

VOCAB_SIZE = 33000
SEED = 67
random.seed(SEED)
torch.manual_seed(SEED)

KNOWN_COUNTS = {"train_supreme_court.json": 55549, "train_acts.json": 18826,
                "train_high_courts.json": 2536738, "train_tribunals.json": 869132}

HF_TOK_REPO = f"{HfApi(token=HF_TOKEN).whoami()['name']}/legal-bpe-tokenizer-phase2-v1"

OUTPUT_DIR = "/kaggle/working/cs781_phase2" if os.path.exists("/kaggle") else "./outputs"
TOKENIZER_DIR = os.path.join(OUTPUT_DIR, "custom_bpe_tokenizer_v2")
TOKEN_DATA_DIR = os.path.join(OUTPUT_DIR, "token_data_v3")

CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "model_checkpoint")
PREDICTION_PATH = os.path.join(OUTPUT_DIR, "prediction.json")
SUBMISSION_ZIP = os.path.join(OUTPUT_DIR, "submission.zip")
LOSS_PLOT_PATH = os.path.join(OUTPUT_DIR, "training_validation_loss.png")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(TOKENIZER_DIR, exist_ok=True)
os.makedirs(TOKEN_DATA_DIR, exist_ok=True)


TRAIN_BIN = os.path.join(TOKEN_DATA_DIR, "train.bin")
VAL_BIN   = os.path.join(TOKEN_DATA_DIR, "val.bin")


In [ ]:

from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders, processors
from transformers import PreTrainedTokenizerFast
from huggingface_hub import hf_hub_download

def download_data():
    local_train_paths = []
    print("\nDownloading phase 2 training data.")
    for tf in TRAIN_FILES:
        try:
            p = hf_hub_download(repo_id = HF_DATASET_REPO, filename = tf, repo_type = "dataset", token = HF_TOKEN)
            local_train_paths.append(p)
            print(f"Downloaded: {tf}")
        except Exception as e:
            print(f"Warning downloading {tf}: {e}")
            
    test_path = None
    for tf in TEST_FILES:
        try:
            test_path = hf_hub_download(repo_id = HF_DATASET_REPO, filename = tf, repo_type = "dataset", token = HF_TOKEN)
            print(f"Downloaded Test File: {tf}")
            break
        except Exception:
            continue
            
    return local_train_paths, test_path

In [ ]:
train_paths, test_path = download_data()

# inspect_json_files(train_paths, n=2)

In [ ]:
P = {os.path.basename(p): p for p in train_paths}
TEST = test_path
print(P.keys(), TEST)

In [ ]:
%%writefile bpe_tokenizer.py
#!/usr/bin/env python3
import os, json, regex
from typing import Dict, List, Tuple
from collections import Counter, defaultdict
from tqdm.auto import tqdm

import regex


_PAT = regex.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")


class PureByteLevelBPETokenizer:
    def __init__(self, vocab_size: int = 33000):
        self.vocab_size = vocab_size
        self.special_tokens = ["<pad>", "<unk>", "<s>", "</s>", "<mask>"]

        self.encoder: Dict[bytes, int] = {}
        self.decoder: Dict[int, bytes] = {}
        self.merges: Dict[Tuple[bytes, bytes], bytes] = {}       # pair -> merged token
        self.merge_ranks: Dict[Tuple[bytes, bytes], int] = {}    # pair -> order learned (lower = earlier)
        self._cache: Dict[str, List[int]] = {}

        self._init_base_vocab()

    def _init_base_vocab(self):
        for b in range(256):
            b_bytes = bytes([b])
            self.encoder[b_bytes] = b
            self.decoder[b] = b_bytes

        # special tokens
        for idx, token in enumerate(self.special_tokens, start=256):
            b_bytes = token.encode("utf-8")
            self.encoder[b_bytes] = idx
            self.decoder[idx] = b_bytes

    @property
    def pad_token_id(self) -> int:
        return self.encoder["<pad>".encode("utf-8")]

    @property
    def unk_token_id(self) -> int:
        return self.encoder["<unk>".encode("utf-8")]

    @property
    def bos_token_id(self) -> int:
        return self.encoder["<s>".encode("utf-8")]

    @property
    def eos_token_id(self) -> int:
        return self.encoder["</s>".encode("utf-8")]

    # ------------------------------------------------------------------ training
    @staticmethod
    def _merge_word(word, a, b, new):
        out, i, n = [], 0, len(word)
        while i < n:
            if i < n - 1 and word[i] == a and word[i + 1] == b:
                out.append(new)
                i += 2
            else:
                out.append(word[i])
                i += 1
        return out

    def train_from_texts(self, texts: List[str], target_vocab_size: int = None, min_word_freq: int = 2):
        if target_vocab_size is None:
            target_vocab_size = self.vocab_size
        num_merges = target_vocab_size - len(self.encoder)
        print(f"\nTraining byte level BPE from scratch. Target merges: {num_merges:,} | Base Vocab: {len(self.encoder)}")

        # 1) pre-tokenize + count unique words (the BPE then runs on this small table, not the raw text)
        word_counts = Counter()
        for text in tqdm(texts, desc="Counting pre-tokens"):
            if text and text.strip():
                word_counts.update(_PAT.findall(text))

        words, freqs = [], []
        for w, c in word_counts.items():
            if c >= min_word_freq:
                words.append([bytes([b]) for b in w.encode("utf-8", "replace")])
                freqs.append(c)
        del word_counts
        print(f"Unique words used for merge learning: {len(words):,}")

        # 2) initial pair statistics (updated incrementally afterwards)
        pair_counts = defaultdict(int)
        where = defaultdict(set)
        for i, w in enumerate(tqdm(words, desc="Initial pair counts")):
            for p in zip(w, w[1:]):
                pair_counts[p] += freqs[i]
                where[p].add(i)

        # 3) merge loop
        pbar = tqdm(total=num_merges, desc="Learning BPE Merges", dynamic_ncols=True)
        for _ in range(num_merges):
            if not pair_counts:
                print("No more candidate pairs found to merge.")
                break

            best_pair, freq = max(pair_counts.items(), key=lambda kv: kv[1])
            if freq < 2:
                print(f"Stopping early: Most frequent pair frequency is {freq}")
                break

            a, b = best_pair
            merged_token = a + b
            self.merges[best_pair] = merged_token
            self.merge_ranks[best_pair] = len(self.merge_ranks)
            if merged_token not in self.encoder:          # two merge paths can give the same bytes
                new_id = len(self.encoder)
                self.encoder[merged_token] = new_id
                self.decoder[new_id] = merged_token

            for i in list(where[best_pair]):
                w, f = words[i], freqs[i]
                for p in zip(w, w[1:]):
                    pair_counts[p] -= f
                    if pair_counts[p] <= 0:
                        del pair_counts[p]
                w = self._merge_word(w, a, b, merged_token)
                words[i] = w
                for p in zip(w, w[1:]):
                    pair_counts[p] += f
                    where[p].add(i)
            del where[best_pair]
            pair_counts.pop(best_pair, None)

            pbar.update(1)
            pbar.set_postfix({"vocab_size": len(self.encoder), "top_pair_freq": freq})

        pbar.close()
        self._cache.clear()
        print(f"Training finished! Final Vocab Size: {len(self.encoder):,}")

    # ------------------------------------------------------------------ encode / decode
    def _encode_word(self, word: str) -> List[int]:
        cached = self._cache.get(word)
        if cached is not None:
            return cached
        tokens = [bytes([b]) for b in word.encode("utf-8", "replace")]
        while len(tokens) >= 2:
            best_pair, best_rank = None, None
            for p in zip(tokens, tokens[1:]):
                r = self.merge_ranks.get(p)
                if r is not None and (best_rank is None or r < best_rank):
                    best_pair, best_rank = p, r
            if best_pair is None:
                break
            tokens = self._merge_word(tokens, best_pair[0], best_pair[1], self.merges[best_pair])
        ids = [self.encoder.get(t, self.unk_token_id) for t in tokens]
        if len(self._cache) < 1_000_000:
            self._cache[word] = ids
        return ids

    def encode(self, text: str) -> List[int]:
        ids: List[int] = []
        for word in _PAT.findall(text):
            ids.extend(self._encode_word(word))
        return ids

    def decode(self, token_ids: List[int]) -> str:
        raw_bytes = b"".join(self.decoder.get(i, b"<unk>") for i in token_ids)
        return raw_bytes.decode("utf-8", errors="replace")

    # ------------------------------------------------------------------ save / load
    def save(self, save_dir: str):
        os.makedirs(save_dir, exist_ok=True)
        vocab_path = os.path.join(save_dir, "vocab.json")
        merges_path = os.path.join(save_dir, "merges.json")
        config_path = os.path.join(save_dir, "tokenizer_config.json")

        # latin1 maps bytes 1:1 onto code points, so any byte string survives JSON
        ser_vocab = {k.decode("latin1"): v for k, v in self.encoder.items()}
        with open(vocab_path, "w", encoding="utf-8") as f:
            json.dump(ser_vocab, f, ensure_ascii=False, indent=2)

        # JSON (not a space-separated text file): tokens can contain spaces, newlines, \r, \xa0 ...
        ser_merges = [[p1.decode("latin1"), p2.decode("latin1")] for (p1, p2) in self.merges]  # in learned order
        with open(merges_path, "w", encoding="utf-8") as f:
            json.dump(ser_merges, f, ensure_ascii=False)

        config = {
            "tokenizer_class": "PureByteLevelBPETokenizer",
            "vocab_size": len(self.encoder),
            "special_tokens": self.special_tokens,
            "pad_token_id": self.pad_token_id,
            "bos_token_id": self.bos_token_id,
            "eos_token_id": self.eos_token_id
        }
        with open(config_path, "w", encoding="utf-8") as f:
            json.dump(config, f, indent=2)

        print(f"Tokenizer saved successfully to {save_dir}")

    @classmethod
    def load(cls, save_dir: str) -> "PureByteLevelBPETokenizer":
        tok = cls()
        with open(os.path.join(save_dir, "vocab.json"), "r", encoding="utf-8") as f:
            ser_vocab = json.load(f)
        tok.encoder = {k.encode("latin1"): v for k, v in ser_vocab.items()}
        tok.decoder = {v: k.encode("latin1") for k, v in ser_vocab.items()}

        tok.merges, tok.merge_ranks = {}, {}
        with open(os.path.join(save_dir, "merges.json"), "r", encoding="utf-8") as f:
            for rank, (s1, s2) in enumerate(json.load(f)):
                p1, p2 = s1.encode("latin1"), s2.encode("latin1")
                tok.merges[(p1, p2)] = p1 + p2
                tok.merge_ranks[(p1, p2)] = rank
        return tok


In [ ]:
%%writefile phase2_finetune.py
#!/usr/bin/env python3
"""Phase 2 legal NER: partial fine-tune of the Phase-1 Llama LM on ALL labelled data
(fold1 + fold2), then predict fold3 and write prediction.json + submission.zip.

Needs next to this file:
  llama_scratch.py   (Config, LlamaLM)                  - your model code
  bpe_tokenizer.py   (PureByteLevelBPETokenizer, _PAT)  - your Phase-1 tokenizer code

SMOKE TEST (a couple of minutes, tiny random model, few docs, exercises every stage):
  python phase2_finetune.py --smoke --tok_dir ./tok_phase1 --data_dir ./data

FINAL RUN (all data, no holdout, fixed epochs):
  python phase2_finetune.py --ckpt final_model.pt --tok_dir ./tok_phase1 --data_dir ./data --epochs 5

Add --val_frac 0.1 if you want a held-out split + best-epoch selection instead.
Add --download to fetch the fold files from the HF dataset (needs HF_TOKEN).
"""
import argparse, bisect, json, math, os, random, sys, time, zipfile
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F

LABELS = ["APP", "RESP", "A.COUNSEL", "R.COUNSEL", "JUDGE", "WIT",
          "AUTH", "COURT", "STAT", "PREC", "DATE", "CASENO"]
TAGS = ["O"] + [f"{p}-{l}" for l in LABELS for p in "BI"]
T2I = {t: i for i, t in enumerate(TAGS)}
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
FILES = {"fold1-text.json": "Phase_2_LNER/Train/fold1-text.json",
         "fold1-gold.json": "Phase_2_LNER/Train/fold1-gold.json",
         "fold2-text.json": "Phase_2_LNER/Train/fold2-text.json",
         "fold2-gold.json": "Phase_2_LNER/Train/fold2-gold.json",
         "fold3-text.json": "Phase_2_LNER/Test/fold3-text.json"}


def log(*a): print(time.strftime("[%H:%M:%S]"), *a, flush=True)


# ------------------------------------------------------------------ data
def get_data_paths(args):
    P = {}
    for name, remote in FILES.items():
        local = os.path.join(args.data_dir, name)
        if not os.path.exists(local) and args.download:
            from huggingface_hub import hf_hub_download
            local = hf_hub_download(HF_DATASET_REPO, remote, repo_type="dataset",
                                    token=os.environ.get("HF_TOKEN"))
        if not os.path.exists(local):
            sys.exit(f"Missing {local} (use --download or put the files in --data_dir)")
        P[name] = local
    return P


def make_tokenizer_fns(tok, PAT):
    def encode_offsets(text):
        ids, offs = [], []
        for m in PAT.finditer(text):
            w, b0 = m.group(), m.start()
            b2c = []
            for ci, ch in enumerate(w): b2c += [ci] * len(ch.encode("utf-8"))
            pos = 0
            for t in tok._encode_word(w):
                nb = len(tok.decoder[t])
                s_c = b2c[min(pos, len(b2c) - 1)]
                e_c = b2c[min(pos + nb, len(b2c)) - 1] + 1
                pos += nb
                while s_c < e_c and w[s_c].isspace(): s_c += 1
                ids.append(t); offs.append((b0 + s_c, b0 + e_c))
        return ids, offs
    return encode_offsets


def bio(offs, ents):
    y = [0] * len(offs); starts = [a for a, _ in offs]
    for s, e, l, *_ in sorted(ents):
        i = bisect.bisect_left(starts, s); first = True
        while i < len(offs) and offs[i][1] <= e:
            a, b = offs[i]
            if a < b and a >= s:
                y[i] = T2I[("B-" if first else "I-") + l]; first = False
            i += 1
    return y


def decode_tags(tag_ids, offs):
    spans, cur = [], None
    for t, (a, b) in zip(tag_ids, offs):
        if a == b: continue
        tag = TAGS[t]
        if tag == "O":
            if cur: spans.append(cur); cur = None
        elif tag[0] == "B" or cur is None or cur[2] != tag[2:]:
            if cur: spans.append(cur)
            cur = [a, b, tag[2:]]
        else: cur[1] = b
    if cur: spans.append(cur)
    return spans


def windows(n, L=512, stride=384):
    out, st = [], 0
    while True:
        out.append((st, min(st + L, n)))
        if st + L >= n: break
        st += stride
    return out


# ------------------------------------------------------------------ official-style scoring / validation
def validate_spans(x, where):
    """Mirrors evaluate_phase2.py format rules."""
    seen = set()
    for j, s in enumerate(x):
        assert isinstance(s, list) and len(s) in (3, 4), f"{where}[{j}] bad shape"
        a, b, l = s[:3]
        assert type(a) is int and type(b) is int and 0 <= a < b, f"{where}[{j}] bad offsets"
        assert l in LABELS, f"{where}[{j}] bad label {l!r}"
        assert len(s) == 3 or isinstance(s[3], str), f"{where}[{j}] bad text"
        assert (a, b, l) not in seen, f"{where}[{j}] duplicate"
        seen.add((a, b, l))


def strict_f1(preds, golds):
    per = {l: [0, 0, 0] for l in LABELS}
    for d, g in golds.items():
        P = {tuple(x[:3]) for x in preds.get(d, [])}; G = {tuple(x[:3]) for x in g}
        for l in LABELS:
            p = {x for x in P if x[2] == l}; q = {x for x in G if x[2] == l}; tp = len(p & q)
            per[l][0] += tp; per[l][1] += len(p) - tp; per[l][2] += len(q) - tp
    rows = {}
    for l, (tp, fp, fn) in per.items():
        p = tp / (tp + fp) if tp + fp else 0; r = tp / (tp + fn) if tp + fn else 0
        rows[l] = (p, r, 2 * p * r / (p + r) if p + r else 0)
    macro = tuple(float(np.mean([v[i] for v in rows.values()])) for i in range(3))
    return macro, rows


# ------------------------------------------------------------------ model
class NER(nn.Module):
    def __init__(s, base, n):
        super().__init__(); s.base = base; s.drop = nn.Dropout(0.1)
        s.head = nn.Linear(base.cfg.d_model, n); s.first_train = 0

    def hidden(s, idx, collect=False):
        b = s.base; T = idx.shape[1]; x = b.tok_emb(idx)
        cos, sin = b.rope_cos[:T], b.rope_sin[:T]; hs = [x]
        for i, l in enumerate(b.layers):
            if i < s.first_train:
                with torch.no_grad(): x, _ = l(x, cos, sin, None, None, None, False)
            else: x, _ = l(x, cos, sin, None, None, None, False)
            if collect: hs.append(x)
        x = b.norm(x)
        return (x, hs) if collect else x

    def forward(s, idx): return s.head(s.drop(s.hidden(idx)))


def autocast(dev):
    return torch.autocast(device_type=dev.type, dtype=torch.float16, enabled=(dev.type == "cuda"))


@torch.no_grad()
def layer_scores(model, wins, dev, seed, n=64):
    model.eval(); sims = []
    for ids, _ in random.Random(seed).sample(wins, min(n, len(wins))):
        x = torch.tensor([ids], device=dev)
        with autocast(dev): _, hs = model.hidden(x, collect=True)
        H = F.normalize(torch.stack([h[0].float().mean(0) for h in hs]), dim=-1)
        sims.append((H @ H.T).cpu())
    S = torch.stack(sims).mean(0); L = S.shape[0] - 1
    return ((S.sum(1) - 1) / L)[1:]


def select_block(score, k):
    L = len(score); k = min(k, L)
    s = max(range(L - k + 1), key=lambda i: score[i:i + k].sum().item())
    return list(range(s, s + k))


@torch.no_grad()
def predict_doc(model, ids, offs, dev):
    if len(ids) == 0: return []
    model.eval(); n = len(ids); acc = torch.zeros(n, len(TAGS)); cnt = torch.zeros(n)
    for a, b in windows(n):
        with autocast(dev):
            pr = model(torch.tensor([ids[a:b]], device=dev))[0].float().softmax(-1).cpu()
        acc[a:b] += pr; cnt[a:b] += 1
    return decode_tags((acc / cnt[:, None].clamp(min=1)).argmax(-1).tolist(), offs)


def evaluate(model, docs, keys, dev):
    preds = {k: predict_doc(model, docs[k]["ids"], docs[k]["offs"], dev) for k in keys}
    return strict_f1(preds, {k: [tuple(e[:3]) for e in docs[k]["ents"]] for k in keys})


# ------------------------------------------------------------------ main
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--smoke", action="store_true")
    ap.add_argument("--ckpt", default=None, help="Phase-1/2 LM checkpoint (final_model.pt); omitted in smoke => random tiny model")
    ap.add_argument("--tok_dir", required=True)
    ap.add_argument("--data_dir", default="./data")
    ap.add_argument("--download", action="store_true")
    ap.add_argument("--test_path", default=None, help="override fold3-text.json path")
    ap.add_argument("--out_dir", default="./out")
    ap.add_argument("--epochs", type=int, default=5)
    ap.add_argument("--bs", type=int, default=8)
    ap.add_argument("--lr_bb", type=float, default=5e-5)
    ap.add_argument("--lr_head", type=float, default=1e-3)
    ap.add_argument("--k_layers", type=int, default=4)
    ap.add_argument("--o_weight", type=float, default=0.3)
    ap.add_argument("--val_frac", type=float, default=0.0, help="0 = train on everything (final run)")
    ap.add_argument("--seed", type=int, default=67)
    args = ap.parse_args()

    if args.smoke:
        args.epochs = 1; args.bs = 4
        if args.val_frac == 0: args.val_frac = 0.3   # exercise the eval path too
    random.seed(args.seed); np.random.seed(args.seed); torch.manual_seed(args.seed)
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    os.makedirs(args.out_dir, exist_ok=True)
    log(f"device={dev} smoke={args.smoke}")

    # tokenizer + model code (your own modules)
    sys.path.insert(0, os.getcwd())
    try:
        from llama_scratch import Config, LlamaLM, PRESETS
        from bpe_tokenizer import PureByteLevelBPETokenizer, _PAT
    except ImportError as e:
        sys.exit(f"Import failed: {e}. Put llama_scratch.py and bpe_tokenizer.py next to this script.")
    tok = PureByteLevelBPETokenizer.load(args.tok_dir)
    encode_offsets = make_tokenizer_fns(tok, _PAT)

    # LM
    if args.ckpt:
        ck = torch.load(args.ckpt, map_location="cpu")
        cfg = Config(**ck["cfg"]); cfg.attn_impl = "sdpa"
        base = LlamaLM(cfg); base.load_state_dict(ck["model"]); del ck
    elif args.smoke:
        cfg = Config(vocab_size=33000, max_seq_len=512, rope_cache_len=1024, attn_impl="sdpa", **PRESETS["tiny"])
        base = LlamaLM(cfg)
        log("SMOKE: tiny random-init LM (pipeline check only, scores will be ~0)")
    else:
        sys.exit("--ckpt is required for a real run")
    log(f"LM: layers={cfg.n_layers} d_model={cfg.d_model}")

    # data
    P = get_data_paths(args)
    docs = {}
    max_docs = 8 if args.smoke else None
    for fold in (1, 2):
        T = json.load(open(P[f"fold{fold}-text.json"], encoding="utf-8"))
        G = json.load(open(P[f"fold{fold}-gold.json"], encoding="utf-8"))
        for n, (d, t) in enumerate(T.items()):
            if max_docs and n >= max_docs: break
            ids, offs = encode_offsets(t)
            ents = G.get(d, [])
            docs[f"{fold}_{d}"] = dict(fold=fold, ids=ids, offs=offs, ents=ents, y=bio(offs, ents))
            if (n + 1) % 50 == 0: log(f"tokenized fold{fold}: {n + 1}/{len(T)} docs")
    keys = sorted(docs)
    assert max(len(docs[k]["ids"]) for k in keys) > 0
    assert max(max(d["ids"]) for d in docs.values()) < cfg.vocab_size, "token id >= model vocab: wrong tokenizer for this checkpoint"
    log(f"{len(docs)} labelled docs loaded")

    rng = random.Random(args.seed)
    val_keys = []
    if args.val_frac > 0:
        for f in (1, 2):
            ks = [k for k in keys if docs[k]["fold"] == f]; rng.shuffle(ks)
            val_keys += ks[:max(1, int(len(ks) * args.val_frac))]
    vs = set(val_keys); train_keys = [k for k in keys if k not in vs]
    log(f"train docs={len(train_keys)} val docs={len(val_keys)}")

    # tokenisation <-> gold alignment upper bound (should be ~1.0 on every doc)
    oracle = {k: decode_tags(docs[k]["y"], docs[k]["offs"]) for k in keys}
    (_, _, of1), orows = strict_f1(oracle, {k: [tuple(e[:3]) for e in docs[k]["ents"]] for k in keys})
    log(f"alignment upper-bound macro-F1 = {of1:.4f}")
    if not args.smoke and of1 < 0.97:
        log("WARNING: alignment bound < 0.97 - tokenizer offsets/BIO conversion is losing entities")

    train_win = [(docs[k]["ids"][a:b], docs[k]["y"][a:b]) for k in train_keys
                 for a, b in windows(len(docs[k]["ids"]), L=min(512, cfg.max_seq_len), stride=384)]
    if args.smoke: train_win = train_win[:24]
    log(f"{len(train_win)} training windows")
    assert train_win, "no training windows"

    # NER model + training-free layer selection
    model = NER(base, len(TAGS)).to(dev)
    score = layer_scores(model, train_win, dev, args.seed)
    SEL = select_block(score, args.k_layers)
    log(f"layer scores={[round(x, 3) for x in score.tolist()]} selected={SEL}")
    for p in model.parameters(): p.requires_grad = False
    for i in SEL:
        for p in model.base.layers[i].parameters(): p.requires_grad = True
    for p in list(model.base.norm.parameters()) + list(model.head.parameters()): p.requires_grad = True
    model.first_train = min(SEL)
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    log(f"trainable {tr / 1e6:.2f}M / {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")

    head_p = list(model.head.parameters())
    bb_p = [p for n_, p in model.named_parameters() if p.requires_grad and not n_.startswith("head")]
    opt = torch.optim.AdamW([{"params": bb_p, "lr": args.lr_bb}, {"params": head_p, "lr": args.lr_head}], weight_decay=0.01)
    steps = args.epochs * math.ceil(len(train_win) / args.bs)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[args.lr_bb, args.lr_head], total_steps=steps, pct_start=0.1)
    scaler = torch.amp.GradScaler("cuda", enabled=(dev.type == "cuda"))
    w = torch.ones(len(TAGS), device=dev); w[0] = args.o_weight
    pad = tok.pad_token_id
    best, best_path, last_path = -1, os.path.join(args.out_dir, "ner_best.pt"), os.path.join(args.out_dir, "ner_last.pt")
    
    n_per_ep = math.ceil(len(train_win) / args.bs); t0 = time.time()
    for ep in range(args.epochs):
        model.train(); random.Random(args.seed + ep).shuffle(train_win); tot, nb = 0.0, 0
        for i in range(0, len(train_win), args.bs):
            b = train_win[i:i + args.bs]; L = max(len(x) for x, _ in b)
            X = torch.tensor([x + [pad] * (L - len(x)) for x, _ in b], device=dev)
            Y = torch.tensor([y + [-100] * (L - len(y)) for _, y in b], device=dev)
            with autocast(dev):
                loss = F.cross_entropy(model(X).float().view(-1, len(TAGS)), Y.view(-1), weight=w, ignore_index=-100)
            assert torch.isfinite(loss), "non-finite loss"
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_([p for g in opt.param_groups for p in g["params"]], 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            tot += loss.item(); nb += 1
            if nb % 25 == 0 or nb == n_per_ep:
                done = ep * n_per_ep + nb; el = time.time() - t0
                log(f"ep {ep+1}/{args.epochs} step {nb}/{n_per_ep} loss {loss.item():.4f} | "
                    f"elapsed {el/60:.1f}m | ETA {el/done*(steps-done)/60:.1f}m")
        msg = f"epoch {ep + 1}/{args.epochs} mean loss {tot / nb:.4f}"
        if val_keys:
            (mp, mr, mf), rows = evaluate(model, docs, val_keys, dev)
            msg += f" | val strict macro P {mp:.3f} R {mr:.3f} F1 {mf:.3f}"
            if mf > best: best = mf; torch.save(model.state_dict(), best_path)
        log(msg)
        torch.save(model.state_dict(), last_path)

    if val_keys and os.path.exists(best_path):
        model.load_state_dict(torch.load(best_path, map_location=dev))
        log("per-label val F1: " + str({l: round(v[2], 3) for l, v in evaluate(model, docs, val_keys, dev)[1].items()}))
    else:
        log("no holdout: using final-epoch weights")

    # predict fold3
    test_path = args.test_path or P["fold3-text.json"]
    T = json.load(open(test_path, encoding="utf-8"))
    if args.smoke: T = dict(list(T.items())[:2])
    out = {}
    for n, (d, t) in enumerate(T.items()):
        if n % 50 == 0: log(f"predicting {n}/{len(T)}")
        ids, offs = encode_offsets(t)
        spans = {(s, e, l) for s, e, l in predict_doc(model, ids, offs, dev) if 0 <= s < e <= len(t)}
        out[d] = [[s, e, l, t[s:e]] for s, e, l in sorted(spans)]
        validate_spans(out[d], f"pred[{d}]")
    assert set(out) == set(T), "id mismatch"
    pred_path = os.path.join(args.out_dir, "prediction.json")
    json.dump(out, open(pred_path, "w", encoding="utf-8"), ensure_ascii=False)
    zip_path = os.path.join(args.out_dir, "submission.zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z: z.write(pred_path, arcname="prediction.json")
    # reload as strict JSON to make sure the file the grader sees is valid
    json.load(open(pred_path, encoding="utf-8"))
    log(f"OK: {len(out)} docs, {sum(len(v) for v in out.values())} entities -> {zip_path}")
    if args.smoke: log("SMOKE TEST PASSED (all stages ran end to end)")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile llama_scratch.py
"""
Llama-style decoder-only LM, written from scratch (no HF / no llama library).

Pieces: RMSNorm, RoPE (or ALiBi), grouped-query attention (GQA; MQA = n_kv_heads 1,
MHA = n_kv_heads == n_heads), SwiGLU MLP, pre-norm residual blocks, tied embeddings,
KV-cache generation with left-padded batches.

Only torch tensors/autograd/nn.Linear/nn.Embedding are used as building blocks.
"""
import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


# ----------------------------------------------------------------------------- config
@dataclass
class Config:
    vocab_size: int = 33000
    d_model: int = 1024
    n_layers: int = 16
    n_heads: int = 16
    n_kv_heads: int = 4          # 16 query heads share 4 KV heads (4 queries per KV head)
    ffn_hidden: int = 2560
    max_seq_len: int = 1024      # training context
    rope_cache_len: int = 4096   # RoPE table length (>= prompt + generated tokens)
    rope_theta: float = 10000.0
    norm_eps: float = 1e-5
    tie_embeddings: bool = True
    pos_enc: str = "rope"        # "rope" | "alibi"
    attn_impl: str = "manual"    # "manual" (hand-written softmax(QK^T)V) | "sdpa" (torch fused kernel, same math)


PRESETS = {
    # exact totals with vocab 33000, tied embeddings
    "target_150m": dict(d_model=896, n_layers=14, n_heads=14, n_kv_heads=2, ffn_hidden=2304), # ~149.8M 
    "base":  dict(d_model=1024, n_layers=16, n_heads=16, n_kv_heads=4, ffn_hidden=2560),  # ~201.6M
    "small": dict(d_model=1024, n_layers=12, n_heads=16, n_kv_heads=4, ffn_hidden=2816),  # ~169.1M
    "tiny":  dict(d_model=64,   n_layers=2,  n_heads=4,  n_kv_heads=2, ffn_hidden=128),   # smoke tests
}


def analytic_param_count(c: Config) -> int:
    """Closed-form parameter count (use it to size a model before building it)."""
    hd = c.d_model // c.n_heads
    attn = 2 * c.d_model * c.n_heads * hd + 2 * c.d_model * c.n_kv_heads * hd   # Wq,Wo + Wk,Wv
    mlp = 3 * c.d_model * c.ffn_hidden                                           # gate, up, down
    per_layer = attn + mlp + 2 * c.d_model                                       # + 2 RMSNorm
    total = per_layer * c.n_layers + c.vocab_size * c.d_model + c.d_model        # + embedding + final norm
    if not c.tie_embeddings:
        total += c.vocab_size * c.d_model
    return total


def count_params(model, verbose=True) -> int:
    """model.parameters() yields a tied (shared) tensor only once, so tied weights are counted once."""
    total = sum(p.numel() for p in model.parameters())
    if verbose:
        emb = model.tok_emb.weight.numel()
        blocks = sum(p.numel() for p in model.layers.parameters())
        print(f"total {total/1e6:.1f}M | transformer blocks {blocks/1e6:.1f}M | embedding {emb/1e6:.1f}M "
              f"({'tied with output head' if model.cfg.tie_embeddings else 'output head separate'}) "
              f"| analytic {analytic_param_count(model.cfg)/1e6:.1f}M")
    return total


# ----------------------------------------------------------------------------- RMSNorm
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        xf = x.float()                                   # always normalise in fp32
        xf = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return (xf * self.weight.float()).type_as(x)


# ----------------------------------------------------------------------------- RoPE
def build_rope_cache(head_dim, max_len, theta):
    inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))   # (hd/2,)
    t = torch.arange(max_len).float()
    freqs = torch.outer(t, inv_freq)                                                # (max_len, hd/2)
    return freqs.cos(), freqs.sin()


def apply_rope(x, cos, sin):
    """x: (B, H, T, hd); cos/sin: (T, hd/2). Rotates the two halves of each head vector (Llama layout)."""
    half = x.shape[-1] // 2
    xf = x.float()
    x1, x2 = xf[..., :half], xf[..., half:]
    cos = cos[None, None]
    sin = sin[None, None]
    out = torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)
    return out.type_as(x)


# ----------------------------------------------------------------------------- ALiBi
def get_alibi_slopes(n_heads):
    def pow2(n):
        start = 2 ** (-8.0 / n)
        return [start ** (i + 1) for i in range(n)]
    if math.log2(n_heads).is_integer():
        s = pow2(n_heads)
    else:
        c = 2 ** math.floor(math.log2(n_heads))
        s = pow2(c) + pow2(2 * c)[0::2][: n_heads - c]
    return torch.tensor(s, dtype=torch.float32)


# ----------------------------------------------------------------------------- GQA
def repeat_kv(x, n_rep):
    """(B, n_kv, S, hd) -> (B, n_kv*n_rep, S, hd); query head h uses kv head h // n_rep."""
    if n_rep == 1:
        return x
    B, n_kv, S, hd = x.shape
    return x[:, :, None].expand(B, n_kv, n_rep, S, hd).reshape(B, n_kv * n_rep, S, hd)


class GQAttention(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        assert cfg.d_model % cfg.n_heads == 0 and cfg.n_heads % cfg.n_kv_heads == 0
        self.nh, self.nkv = cfg.n_heads, cfg.n_kv_heads
        self.hd = cfg.d_model // cfg.n_heads
        self.n_rep = self.nh // self.nkv
        self.impl = cfg.attn_impl
        self.wq = nn.Linear(cfg.d_model, self.nh * self.hd, bias=False)
        self.wk = nn.Linear(cfg.d_model, self.nkv * self.hd, bias=False)
        self.wv = nn.Linear(cfg.d_model, self.nkv * self.hd, bias=False)
        self.wo = nn.Linear(self.nh * self.hd, cfg.d_model, bias=False)

    def forward(self, x, cos, sin, alibi_slopes, key_valid, past, use_cache):
        B, T, _ = x.shape
        q = self.wq(x).view(B, T, self.nh, self.hd).transpose(1, 2)    # (B, nh,  T, hd)
        k = self.wk(x).view(B, T, self.nkv, self.hd).transpose(1, 2)   # (B, nkv, T, hd)
        v = self.wv(x).view(B, T, self.nkv, self.hd).transpose(1, 2)

        if cos is not None:                                            # RoPE on q and k (not v)
            q = apply_rope(q, cos, sin)
            k = apply_rope(k, cos, sin)

        if past is not None:                                           # cache holds only nkv heads
            k = torch.cat([past[0], k], dim=2)
            v = torch.cat([past[1], v], dim=2)
        new_cache = (k, v) if use_cache else None
        S = k.shape[2]

        k = repeat_kv(k, self.n_rep)
        v = repeat_kv(v, self.n_rep)

        # relative distance of every (query, key) pair; keys j <= query position are visible
        qpos = torch.arange(S - T, S, device=x.device)
        kpos = torch.arange(S, device=x.device)
        rel = qpos[:, None] - kpos[None, :]                            # (T, S)
        allowed = (rel >= 0)[None, None]                               # (1, 1, T, S)
        if key_valid is not None:                                      # hide left-padding keys
            allowed = allowed & key_valid[:, None, None, :]
        bias = None
        if alibi_slopes is not None:
            bias = -alibi_slopes.view(1, -1, 1, 1) * rel.clamp(min=0).float()[None, None]   # (1, nh, T, S)

        if self.impl == "sdpa" and key_valid is None and T == S:      # training fast path
            if bias is None:
                out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
            else:
                m = bias.masked_fill(~allowed, torch.finfo(q.dtype).min).to(q.dtype)
                out = F.scaled_dot_product_attention(q, k, v, attn_mask=m)
        else:                                                          # hand-written attention
            scores = ((q * (self.hd ** -0.5)) @ k.transpose(-2, -1)).float()   # (B, nh, T, S)
            if bias is not None:
                scores = scores + bias
            scores = scores.masked_fill(~allowed, -1e30)               # finite, so a fully-masked pad row can't give NaN
            probs = torch.softmax(scores, dim=-1).to(v.dtype)
            out = probs @ v                                            # (B, nh, T, hd)

        out = out.transpose(1, 2).reshape(B, T, self.nh * self.hd)
        return self.wo(out), new_cache


# ----------------------------------------------------------------------------- SwiGLU
class SwiGLU(nn.Module):
    def __init__(self, d_model, hidden):
        super().__init__()
        self.w_gate = nn.Linear(d_model, hidden, bias=False)
        self.w_up = nn.Linear(d_model, hidden, bias=False)
        self.w_down = nn.Linear(hidden, d_model, bias=False)

    def forward(self, x):
        return self.w_down(F.silu(self.w_gate(x)) * self.w_up(x))


# ----------------------------------------------------------------------------- block + model
class Block(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.attn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.attn = GQAttention(cfg)
        self.mlp_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.mlp = SwiGLU(cfg.d_model, cfg.ffn_hidden)

    def forward(self, x, cos, sin, alibi_slopes, key_valid, past, use_cache):
        h, new_cache = self.attn(self.attn_norm(x), cos, sin, alibi_slopes, key_valid, past, use_cache)
        x = x + h
        x = x + self.mlp(self.mlp_norm(x))
        return x, new_cache


class LlamaLM(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        assert cfg.pos_enc in ("rope", "alibi") and cfg.attn_impl in ("manual", "sdpa")
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.layers = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        if cfg.tie_embeddings:
            self.lm_head.weight = self.tok_emb.weight

        if cfg.pos_enc == "rope":
            cos, sin = build_rope_cache(cfg.d_model // cfg.n_heads, cfg.rope_cache_len, cfg.rope_theta)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)
            self.alibi_slopes = None
        else:
            self.rope_cos = self.rope_sin = None
            self.register_buffer("alibi_slopes", get_alibi_slopes(cfg.n_heads), persistent=False)

        self.apply(self._init_weights)
        for name, p in self.named_parameters():       # GPT-2 style: shrink the residual-branch output projections
            if name.endswith("wo.weight") or name.endswith("w_down.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * cfg.n_layers))

    @staticmethod
    def _init_weights(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None, use_cache=False, cache=None, start_pos=0,
                key_valid=None, all_logits=False):
        """
        training:  model(x, targets=y) -> scalar loss
        inference: model(idx, use_cache=True, cache=..., start_pos=..., key_valid=...) -> (logits, cache)
                   logits are for the last position only unless all_logits=True.
        """
        B, T = idx.shape
        x = self.tok_emb(idx)
        cos = sin = None
        if self.cfg.pos_enc == "rope":
            assert start_pos + T <= self.cfg.rope_cache_len, "sequence longer than the RoPE table"
            cos = self.rope_cos[start_pos:start_pos + T]
            sin = self.rope_sin[start_pos:start_pos + T]

        new_caches = []
        for i, layer in enumerate(self.layers):
            past = cache[i] if cache is not None else None
            x, c = layer(x, cos, sin, self.alibi_slopes, key_valid, past, use_cache)
            new_caches.append(c)
        x = self.norm(x)

        if targets is not None:
            logits = self.lm_head(x)
            return F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.reshape(-1), ignore_index=-100)
        logits = self.lm_head(x if all_logits else x[:, -1:, :])
        return logits, (new_caches if use_cache else None)


# ----------------------------------------------------------------------------- generation
@torch.no_grad()
def generate(model, prompts, max_new_tokens, eos_id, pad_id, temperature=0.1, top_k=30,
             top_p=0.95, rep_penalty=1.2, ban_eos=True):
    """
    prompts: list of lists of token ids (any lengths). Left-pads them into one batch, runs one prefill,
    then decodes with a KV cache. Returns a list of lists of generated ids (EOS / padding removed).
    temperature <= 0 -> greedy.
    """
    model.eval()
    device = next(model.parameters()).device
    V = model.cfg.vocab_size
    B = len(prompts)
    L = max(len(p) for p in prompts)
    ids = torch.full((B, L), pad_id, dtype=torch.long, device=device)
    key_valid = torch.zeros(B, L, dtype=torch.bool, device=device)
    seen = torch.zeros(B, V, dtype=torch.bool, device=device)
    for b, p in enumerate(prompts):
        t = torch.tensor(p, dtype=torch.long, device=device)
        ids[b, L - len(p):] = t
        key_valid[b, L - len(p):] = True
        seen[b, t] = True

    out = [[] for _ in range(B)]
    finished = torch.zeros(B, dtype=torch.bool, device=device)
    amp = torch.autocast(device_type=device.type, dtype=torch.float16, enabled=(device.type == "cuda"))
    with amp:
        logits, cache = model(ids, use_cache=True, key_valid=key_valid, start_pos=0)
        for step in range(max_new_tokens):
            lg = logits[:, -1, :].float()
            if ban_eos:
                lg[:, eos_id] = float("-inf")
            if rep_penalty != 1.0:                                     # CTRL-style repetition penalty
                pen = torch.where(lg > 0, lg / rep_penalty, lg * rep_penalty)
                lg = torch.where(seen, pen, lg)
            if temperature <= 0:
                nxt = lg.argmax(-1)
            else:
                lg = lg / temperature
                if top_k and top_k > 0:
                    kth = torch.topk(lg, min(top_k, V)).values[:, -1:]
                    lg = lg.masked_fill(lg < kth, float("-inf"))
                if top_p < 1.0:
                    sl, si = torch.sort(lg, descending=True, dim=-1)
                    pr = torch.softmax(sl, dim=-1)
                    drop = (pr.cumsum(-1) - pr) > top_p                # keep the token that crosses top_p
                    sl = sl.masked_fill(drop, float("-inf"))
                    lg = torch.full_like(lg, float("-inf")).scatter(1, si, sl)
                nxt = torch.multinomial(torch.softmax(lg, dim=-1), 1).squeeze(1)

            nxt = torch.where(finished, torch.full_like(nxt, pad_id), nxt)
            for b in range(B):
                if not finished[b] and nxt[b].item() != eos_id:
                    out[b].append(int(nxt[b].item()))
            finished = finished | (nxt == eos_id)
            seen.scatter_(1, nxt[:, None], True)
            if bool(finished.all()):
                break
            key_valid = torch.cat([key_valid, torch.ones(B, 1, dtype=torch.bool, device=device)], dim=1)
            logits, cache = model(nxt[:, None], use_cache=True, cache=cache,
                                  start_pos=L + step, key_valid=key_valid)
    return out

In [ ]:
#!cp llama_scratch.py bpe_tokenizer.py phase2_finetune.py

In [ ]:
from huggingface_hub import hf_hub_download
for f in ["vocab.json","merges.json","tokenizer_config.json"]:
    hf_hub_download("DumbLad/legal-bpe-tokenizer-phase1", f, repo_type="model",
                    local_dir="/kaggle/working/tok_phase1", token=HF_TOKEN)

In [ ]:
import shutil, os
os.makedirs("data", exist_ok=True)
for p in list(P.values()) + [TEST]:
    shutil.copy(p, "data/")
print(sorted(os.listdir("data")))

In [ ]:
'''!python phase2_finetune.py --smoke --tok_dir /kaggle/working/tok_phase1 --data_dir ./data --out_dir ./out_smoke'''

In [ ]:
!python -u phase2_finetune.py --ckpt /kaggle/input/datasets/koooshiieee/phase2v1/final_model.pt --tok_dir /kaggle/working/tok_phase1 --data_dir ./data --epochs 5 --out_dir /kaggle/working/out

In [ ]:
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
repo = f"{api.whoami()['name']}/legal-ner-phase2-v1"   # change the name if you like
api.create_repo(repo, repo_type="model", private=True, exist_ok=True)

for f in ["submission.zip", "prediction.json", "ner_last.pt"]:
    api.upload_file(path_or_fileobj=f"/kaggle/working/out/{f}",
                    path_in_repo=f, repo_id=repo, repo_type="model")
print("Uploaded to https://huggingface.co/" + repo)